In [1]:
import sys
sys.path.insert(0, '../')

from src.load_data import PatientLoader
from src.process import preprocess

## Import Data

In [2]:
from nilearn import plotting, image, masking

In [3]:
file_path = "data/sub-1004"
loader = PatientLoader()
recording = loader.load(file_path)

D:\Universidad\6 infierno de Dante\neuroimagen\projecto_git\Neuroimage-FinalProjectCode\src\load_data.py:51: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  mni_nifti = image.resample_to_img(
D:\Universidad\6 infierno de Dante\neuroimagen\projecto_git\Neuroimage-FinalProjectCode\src\load_data.py:51: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  mni_nifti = image.resample_to_img(
D:\Universidad\6 infierno de Dante\neuroimagen\projecto_git\Neuroimage-FinalProjectCode\src\load_data.py:51: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  mni_nifti = image.resample_to_img(
D:\Universidad\6 infierno de 

## Connectivity analysis

https://nilearn.github.io/dev/auto_examples/04_glm_first_level/plot_adhd_dmn.html
https://medium.com/data-science/using-neural-networks-for-a-functional-connectivity-classification-of-fmri-data-ff0999057bc6

In [4]:
import numpy as np

from nilearn import plotting
from nilearn.glm.first_level import (
    FirstLevelModel,
    make_first_level_design_matrix,
)
from nilearn.maskers import NiftiSpheresMasker

Extract the seed region's time course

In [5]:
# Prepare seed
pcc_coords = (0, -53, 26)

In [ ]:
seed_masker = NiftiSpheresMasker(
    [pcc_coords],
    radius=10,
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=recording.func.sampling_period, #sampling period of the data
    memory="nilearn_cache",
    memory_level=1,
    verbose=1,
)

seed_time_series = seed_masker.fit_transform(recording.func.img)

n_scans = seed_time_series.shape[0]
frametimes = np.linspace(0, (n_scans - 1) * recording.func.sampling_period, n_scans)

Estimate contrasts

In [ ]:
design_matrix = make_first_level_design_matrix(
    frametimes,
    hrf_model="spm",
    add_regs=seed_time_series,
    add_reg_names=["pcc_seed"],
)
dmn_contrast = np.array([1] + [0] * (design_matrix.shape[1] - 1))
contrasts = {"seed_based_glm": dmn_contrast}

Perform first level analysis

In [ ]:
first_level_model = FirstLevelModel(verbose=1)
first_level_model = first_level_model.fit(
    run_imgs=recording.func.img, design_matrices=design_matrix
)

In [ ]:
print("Contrast seed_based_glm computed.")
z_map = first_level_model.compute_contrast(
    contrasts["seed_based_glm"], output_type="z_score"
)

In [ ]:
display = plotting.plot_stat_map(
    z_map, threshold=3.0, title="Seed based GLM", cut_coords=pcc_coords
)
display.add_markers(
    marker_coords=[pcc_coords], marker_color="g", marker_size=300
)

In [ ]:
report = first_level_model.generate_report(
    contrasts=contrasts,
    title="ADHD DMN Report",
    cluster_threshold=15,
    min_distance=8.0,
    plot_type="glass",
)

In [ ]:
report